In [26]:
!pip install pvlib

In [27]:
!pip install pymoo

In [28]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import fsolve
import time
import sys
import pvlib
from pvlib.solarposition import get_solarposition
from pvlib.location import Location
from pvlib.atmosphere import get_relative_airmass
import time
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.termination import get_termination
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM

#------------------------------------------------------------
start_time = time.time()

#________________________________________PV Modeling_______________________________________________________________________
# Constants
k = 1.38e-23  # Boltzmann constant (J/K)
q = 1.6e-19   # Charge of electron (C)
Ns = 60       # Number of cells in series
Np = 1        # Number of parallel strings


# PV parameters
Iph_ref = 7.7   # Reference photo-generated current (A) for our pv
Rs = 0.5       # Series resistance (Ω)
Rsh = 800      # Shunt resistance (Ω)
Is_ref1 = 5e-15   # Saturation current for diode 1 (A)
Is_ref2 = 5e-15   # Saturation current for diode 2 (A)
Is_refi = 5e-14   # Saturation current due to interface defects (A)
n1 = 1.1      # Ideality factor for diode 1 - Auger dominated SHJ
n2 = 1.0      # Ideality factor for diode 2
ni = 0.85  # Ideality factor for interface defects
Voc_ref_per_cell = 0.751  # Open-circuit voltage per cell (V)
Eg = 1.12               # Silicon bandgap (eV)
T_ref = 25 + 273.15
NOCT=45
Eg_ref= 1.12
alpha = 4.73e-4
rho_contact =2.5e-3 #Ω*cm2 (p-nc-Si:H/HCO contact resistancy)
A_cell_pv=243 #cm2
# Essential missing parameters for SHJ (from research papers)
aSi_thickness = 6e-9       # a-Si:H(i) layer thickness (5nm) [1]
TCO_resistivity = 2.5e-4     # ITO sheet resistance (Ω·cm) [7]
V_oc_temp_coeff = -0.24 /100   # SHJ voltage temp coefficient (%/K) [4]
j_sc = Iph_ref / A_cell_pv
N_def = 5e14       # 1e15 m⁻³ defects
pv_module_area = 1.74  # m^2

# Solar and geographical parameters
'''Lund, Sweden → Latitude: 55.7° N, Longitude: 13.2° E
Lund, Czech Republic → Latitude: 50.1° N, Longitude: 14.4° E
Andorra (Andorra la Vella) → Latitude: 42.5° N, Longitude: 1.5° E'''

latitude = 50.1  # Change this to your location
longitude = 14.4  # Change this to your location
tilt_angle_deg = 30  # PV panel tilt angle (adjust as needed)
sun_zenith_deg=45

#---------------------------------------------Modelling-------------------------------------------------------

# Function to calculate effective irradiance including diffuse and reflected components
def calculate_effective_irradiance(G, G_rear, tilt_angle_deg, sun_zenith_deg, bifaciality=0.9,  albedo=0.2):
    tilt_angle_rad = np.radians(tilt_angle_deg)
    sun_zenith_rad = np.radians(sun_zenith_deg)

    # Direct component
    G_direct = G * np.cos(tilt_angle_rad - sun_zenith_rad)

    # Diffuse component (approximation for isotropic model)
    G_diffuse = 0.3 * G  # Assume 30% of total radiation is diffuse

    # Ground-reflected component
    G_reflected = albedo * G * (1 - np.cos(tilt_angle_rad)) / 2

    # Rear-side effective irradiance (assuming similar ground reflection model)
    G_rear_effective = G_rear * bifaciality

    return G_direct + G_diffuse + G_reflected +  G_rear_effective

# Function to calculate bifacial gain
def bifacial_gain(G, G_rear, bifaciality=0.9):
    return G + bifaciality * G_rear

def bandgap_energy(T):
    # Example bandgap calculation, not involving calculate_is
    return Eg_ref * (1 - alpha * (T - T_ref))

'''
# Example correction for a circular recursion
def calculate_is(temperature, Is_ref):
    # Ensure bandgap_energy is calculated without calling calculate_is again
    Eg_T = bandgap_energy(temperature)
    # other calculations that don't involve recursion
    Is = Is_ref * (Eg_T / Eg_ref)  # Adjust as per your model
    return Is
'''

def calculate_iph(Iph_ref, G, T_ambient, G_ref=1000, T_ref=25, alpha_Isc=0.0005):
    return Iph_ref * (G / G_ref) * (1 + alpha_Isc * (T_ambient - T_ref))

# Function to calculate cell temperature from ambient temperature and irradiance
def calculate_cell_temperature(temperature, G, NOCT=45):
    return temperature + ((NOCT - 20) / 800) * G  # More accurate NOCT model


# Function to calculate total resistance including contact resistivity
def calculate_total_resistance(Rs, Rsh, rho_contact, A_cell_pv):
    """Compute total resistance including series, shunt, and contact resistance."""
    R_contact = rho_contact / A_cell_pv  # Convert resistivity to resistance (Ω)
    return Rs + R_contact, Rsh  # Return total series and shunt resistance

def adjust_voc_temp(Voc_ref_per_cell, temperature, T_ref, V_oc_temp_coeff):
    """Adjust open-circuit voltage based on temperature effect."""
    V_oc_temp_coeff_abs = V_oc_temp_coeff / 100  # Convert %/K to fraction/K
    return Voc_ref_per_cell * (1 + V_oc_temp_coeff_abs * (temperature - T_ref))

def enhanced_recombination(voltage, temperature, Rs, Rsh, rho_contact, A_cell_pv, aSi_thickness, TCO_resistivity, ni):
    """Compute recombination current, considering contact resistance, interface defects, and passivation quality."""

    A_cell_m2 = A_cell_pv * 1e-4

    # Contact resistance
    R_contact = rho_contact / A_cell_pv  # Contact resistance (Ohms)
    R_total = Rs + R_contact  # Total resistance (series + contact)

    # Surface recombination velocity (S) based on material thickness
    S = 1e5 * np.exp(-aSi_thickness / 2e-9)  # Surface recombination (m/s)

    Voc_adj = adjust_voc_temp(Voc_ref_per_cell, temperature, T_ref, V_oc_temp_coeff)
    # Temperature-dependent thermal voltage (Vt)
    Vt = (k * (temperature + 273.15)) / q   # Thermal voltage in volts

    # Diode ideality factor, ni (typically around 1 for ideal material)
    exponent = voltage / (ni * Vt)

    # Cap the exponent to avoid overflow (tuning the range here)
    exponent = np.clip(exponent, -10, 10)  # Avoid large values

    # Defect current due to recombination

    defect_current = 1e-12


    # Recombination current using a more realistic approach
    recombination_current = defect_current * (voltage / R_total) * (1 + S / (TCO_resistivity * A_cell_pv))


    return recombination_current

# Function to compute solar position and adjust irradiance
def get_solar_irradiance(time, latitude, longitude, tilt_angle_deg, G=1000):
    location = Location(latitude, longitude)
    solar_position = get_solarposition(time, location.latitude, location.longitude)

    sun_zenith_deg = solar_position['zenith'].values[0]
    effective_irradiance = calculate_effective_irradiance(G, 0.1*G, tilt_angle_deg, sun_zenith_deg)

    return effective_irradiance

def implied_ff(Voc_adj, j_sc, temperature):

    """Empirical relationship for implied fill factor (FF)."""
    Vt = (k * (temperature + 273.15)) / q
    voc_normalized = Voc_adj / (n1 * Vt)  # Dimensionless Voc
    return (voc_normalized - np.log(voc_normalized + 0.72)) / (voc_normalized + 1)


def pv_model(voltage, params, effective_irradiance, rho_contact,  A_cell, aSi_thickness,  TCO_resistivity, ni):
    # Unpack parameters
    Iph_ref, Is_ref1, Is_ref2, Is_refi, Rs, Rsh, n1, n2, ni, temperature, rho_contact = params

    # Adjust the photocurrent for irradiance and temperature
    Iph = calculate_iph(Iph_ref, effective_irradiance, temperature, G_ref=1000, T_ref=25, alpha_Isc=0.0005)

    Voc_adj = adjust_voc_temp(Voc_ref_per_cell, temperature, T_ref, V_oc_temp_coeff)
    # Compute thermal voltage
    Vt = (k * (temperature + 273.15)) / q

    # Compute total resistances
    Rs_total, Rsh_total = calculate_total_resistance(Rs, Rsh, rho_contact, A_cell_pv)

    # Initialize diode currents
    Id_prev = 0

    while True:
        # Temperature-dependent saturation currents
        Is1 = Is_ref1 * ((temperature + 273.15)/T_ref)**3 * np.exp((Eg*q/(n1*k))*(1/T_ref - 1/(temperature + 273.15)))
        Is2 = Is_ref2 * ((temperature + 273.15)/T_ref)**3 * np.exp((Eg*q/(n2*k))*(1/T_ref - 1/(temperature + 273.15)))
        Isi = Is_refi * ((temperature + 273.15)/T_ref)**3 * np.exp((Eg*q/(ni*k))*(1/T_ref - 1/(temperature + 273.15)))

        # Compute diode currents
        Id1 = Is1 * (np.exp((voltage + voltage * Rs_total / Rsh_total) / (n1 * Ns * Vt)) - 1)
        Id2 = Is2 * (np.exp((voltage + voltage * Rs_total / Rsh_total) / (n2 * Ns * Vt)) - 1)
        Idi = Isi * (np.exp((voltage + voltage * Rs_total / Rsh_total) / (ni * Ns * Vt)) - 1)  # Interface defects contribution

        # Total diode current
        Id = Id1 + Id2 + Idi

        # Compute current considering resistances
        I = (Iph * Rsh_total - Id * Rsh_total - voltage) / (Rsh_total + Rs_total)

        # Recombination current including voltage-dependent defect model and contact resistance
        I_rec = enhanced_recombination(voltage, temperature,Rs, Rsh, rho_contact,  A_cell_pv, aSi_thickness, TCO_resistivity, ni)
        #print(I_rec)


        # Calculate total current (subtract recombination losses)
        I_total = I - I_rec   # Subtract recombination current from the total current

        # Convergence check (stop if the difference between iterations is small)
        if abs(Id - Id_prev) < 1e-3 :
            break

        Id_prev = Id

    return I_total



# PV model considering series/parallel configuration
def pv_model_series(voltage, params, effective_irradiance, num_modules_series, num_modules_parallel):
    single_module_voltage = voltage
    single_module_current = pv_model(single_module_voltage, params, effective_irradiance,rho_contact,  A_cell_pv,aSi_thickness, TCO_resistivity, ni)
    total_current = single_module_current * num_modules_parallel
    total_voltage = single_module_voltage * num_modules_series
    return total_current, total_voltage

def perturb_observe_series(params, effective_irradiance, num_modules_series, num_modules_parallel, voltage_start=0, voltage_step=0.001):
    voltages = []
    currents = []
    powers = []
    resistances = []
    voltage = voltage_start
    mppt_index = 0  # Initialize mppt_index as an integer
    max_power = 0

    while True:
        # Pass params_with_temp to pv_model_series
        current, total_voltage = pv_model_series(voltage, params, effective_irradiance, num_modules_series, num_modules_parallel)
        power_output = total_voltage * current
        resistance = total_voltage / current if current != 0 else float('inf')

        voltages.append(total_voltage)
        currents.append(current)
        powers.append(power_output)
        resistances.append(resistance)

        # Track the maximum power output and its index
        if power_output > max_power:
            max_power = power_output
            mppt_index = len(powers) - 1  # Update mppt_index as an integer

        if current <= 0:  # Stop when the current reaches zero
            break

        voltage += voltage_step

    return voltages, currents, powers, mppt_index  # Return mppt_index as an integer

# Find the MPPT point
def find_mppt(voltages, currents, voltage_step=0.001):
    voltage = np.array(voltages)
    current = np.array(currents)
    power = voltage * current
    max_power_index = np.argmax(power)

    left_voltage = voltage[max_power_index] - voltage_step
    right_voltage = voltage[max_power_index] + voltage_step

    left_power = left_voltage * current[max_power_index]
    right_power = right_voltage * current[max_power_index]

    if left_power > right_power:
        new_voltage = left_voltage
    else:
        new_voltage = right_voltage

    return voltage[max_power_index], current[max_power_index], power[max_power_index]

# Example usage
time_now = pd.Timestamp.now()  # Get the current time
G = 1000  # Default global irradiance in W/m²

effective_G = get_solar_irradiance(time_now, latitude, longitude, tilt_angle_deg, G)

print(f"Effective Irradiance at {time_now}: {effective_G} W/m²")

#________________________________________PEM Modeling_______________________________________________________________________
# Constants

def V_cell(T, p_cat, p_an, δ_mem, A_cell, a_an, a_cat, i, i_0_an, i_0_cat, T_ref, i_lim):

    #Global constants
    R = 8.314  # Gas constant in J/(mol·K)
    F= 9.648e+4
    # Constants

    k1 = 8.5e-4
    k2 = R//(F)
    k3 = 6.1078e-3
    k4 = 17.694
    k5 = 34.85
    k6 = 0.1113
    k7 = 3.26e-3
    k8 = 1268
    k9 = R/F

    # Intermediate calculations
    Erev = 1.229 - k1 * (T - 298.15) + ((k2/2) * T * np.log(((p_cat - k3 * np.exp(k4 * ((T - 273.15) / (T - k5)))) * np.square(p_an - k3 * np.exp(k4 * ((T - 273.15) / (T - k5))))) / (k3 * np.exp(k4 * ((T - 273.15) / (T - k5))))))

    Vohm = δ_mem / (A_cell * k6 - k7 * np.exp(k8 * ((1 / 303) - (1 / T))))


    Vact = ((k9 * T) / a_an) * np.log(i /i_0_an) + ((k9 * T) / a_cat) * np.log((i + 1e-10) /(i_0_cat+ 1e-10))


    Vcon = (((k2 * T) / 2*a_an) * np.log(i_lim/(i_lim-1) ))


    # Cell voltage calculation
    V_cell = Erev + Vohm + Vact + Vcon
    #V_cell = 1.229 - k1 * (T - 298.15) + (k2 * T * np.log(ln_term)) + mem_term + (((k9 * T) / α_an) * sin_term) + (((k2 * T) / α_an) * np.log(ln_term2))


    return V_cell

def V_PEM(T, p_cat, p_an, δ_mem, A_cell, a_an, a_cat, i, i_0_an, i_0_cat, T_ref, i_lim, num_cell_series):
    return V_cell(T, p_cat, p_an, δ_mem, A_cell, a_an, a_cat, i, i_0_an, i_0_cat, T_ref, i_lim) * num_cell_series

def I_PEM(i_cell, num_cell_parallel):
    return i_cell * num_cell_parallel



# Define the load fraction points and the corresponding efficiency values
load_points = [0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.70, 0.8, 0.9, 1.0]  # Fractional loads (10%, 50%, 100%)
efficiency_points = [0, 0.70, 0.705, 0.713, 0.72, 0.715, 0.705, 0.70, 0.695, 0.68, 0.67, 0.66]  # Efficiencies at corresponding loads


#________________________________________Perameters_______________________________________________________________________


# Example parameters
params_base = [Iph_ref, Is_ref1, Is_ref2, Is_refi, Rs, Rsh, n1, n2, ni, rho_contact]  # Updated parameter list

#T = 335.15  # K
δ_mem =  0.0175  # Example value for membrane thickness
A_cell = 500 # cm2  Example value for cell area
a_an = 0.9# Example value for anode overpotential coefficient
a_cat = 0.6
i_0_an  = 1.0e-3   # IrO2 anode (well optimized)
i_0_cat = 5.0e-2   # Pt cathode
T_ref = 298.15  # Reference temperature in Kelvin
i_lim = 6.1 # Example value for current limit

p_cat = 35  # Example value for cathode pressure
p_an = 1.0  # Example value for anode pressure
i_cell = A_cell * 1  #---------------------------IMPORTANT!!!!!!!
Faraday_constant = 96485
LHV = 33.3*1000
module_area= 0.5 #m2
efficiency = 0.69
PEM_capacity=42000

#____________________________________________________________________Optimization___________________________________________________________________________________________________________________

import numpy as np

def find_intersection_point(voltages_pv, currents_pv, voltages_pem, currents_pem):
    """
    Find the closest intersection point between PV and PEM voltage-current characteristics.
    """
    min_error = float('inf')
    best_point = None

    for v_pv, i_pv in zip(voltages_pv, currents_pv):
        for v_pem, i_pem in zip(voltages_pem, currents_pem):
            error = np.sqrt((v_pv - v_pem)**2 + (i_pv - i_pem)**2)  # Euclidean distance
            if error < min_error:
                min_error = error
                best_point = (v_pv, i_pv)

    return best_point  # Closest match

Effective Irradiance at 2026-01-08 15:45:47.602551: 834.0252358075967 W/m²


In [29]:
from pymoo.core.problem import Problem
from pymoo.core.variable import Integer
import numpy as np

class PV_PEM_Optimization(Problem):

    def __init__(self, params, irradiance, LHV, module_area, efficiency,
                 tilt_angle_deg, sun_zenith_deg,
                 p_cat, p_an, δ_mem, A_cell_pem, a_an, a_cat,
                 i_0_an, i_0_cat, T_ref, i_lim,
                 cost_pv_module=245, cost_pem_cell=1980):

        # 4 design variables:
        # 0: num_modules_series, 1: num_modules_parallel, 2: num_cell_series, 3: num_cell_parallel
        n_var = 5
        xl = np.array([2, 45, 42, 1, 330])  # Lower bounds
        xu = np.array([2, 90, 42, 1, 360])  # Upper bounds
        super().__init__(n_var=n_var,
                         n_obj=4,   # Hydrogen, STH efficiency, Economic, Energy Loss - Changed from 3 to 4
                         n_constr=1,  # PV-PEM matching
                         xl=xl,
                         xu=xu,
                         type_var=Integer)

        self.params = params
        self.irradiance = irradiance
        #self.T = T
        self.LHV = LHV
        self.module_area = pv_module_area
        self.efficiency = efficiency
        self.tilt_angle_deg = tilt_angle_deg
        self.sun_zenith_deg = sun_zenith_deg
        self.p_cat = p_cat
        self.p_an = p_an
        self.δ_mem = δ_mem
        self.A_cell_pem = A_cell_pem
        self.a_an = a_an
        self.a_cat = a_cat
        self.i_0_an = i_0_an
        self.i_0_cat = i_0_cat
        self.T_ref = T_ref
        self.i_lim = i_lim
        self.cost_pv_module = cost_pv_module
        self.cost_pem_cell = cost_pem_cell

    def _evaluate(self, X, out, *args, **kwargs):
        n_samples = X.shape[0]
        f1 = np.zeros(n_samples)  # Hydrogen production (maximize)
        f2 = np.zeros(n_samples)  # STH efficiency (maximize)
        f3 = np.zeros(n_samples)  # Economic objective (minimize cost)
        f4 = np.zeros(n_samples) # Energy loss
        g1 = np.zeros(n_samples)  # Constraint: PV voltage >= PEM voltage

        for i, x in enumerate(X):
            num_modules_series = int(x[0])
            num_modules_parallel = int(x[1])
            num_cell_series = int(x[2])
            num_cell_parallel = int(x[3])
            PEM_Temperature= int(x[4])

            hydro_results, sth_results, cost_results, loss_results = [], [], [], []

            # --- PV IV Curve ---
            effective_irradiance = calculate_effective_irradiance(
                irradiance, 0.1 * irradiance,
                tilt_angle_deg,sun_zenith_deg
            )
            voltages_pv, currents_pv, powers_pv, mppt_index = perturb_observe_series(
                params=params_base + [temperature], # params_base + current T for pv_model
                effective_irradiance=effective_irradiance,
                num_modules_series=num_modules_series,
                num_modules_parallel=num_modules_parallel
            )
            mppt_voltage, mppt_current, mppt_power = find_mppt(voltages_pv, currents_pv)

            # --- Generate PEM IV curve
            pem_current_densities = np.linspace(0.01, I_PEM(i_cell, num_cell_parallel), 100)  # Current density (A/cm²)
            # Use the parameters from the current hour's results (stored in the result dictionary)
            pem_voltages = [V_PEM(temperature + 273.15, p_cat, p_an, δ_mem,
                                  A_cell, a_an, a_cat,
                                  i, i_0_an, i_0_cat,
                                  T_ref, i_lim, num_cell_series)
                            for i in pem_current_densities]



            # --- Find intersection (operating point) ---
            operating_point = find_intersection_point(voltages_pv, currents_pv, pem_voltages, pem_current_densities)
            if not operating_point:
                continue
            operating_voltage, operating_current = operating_point

            # Inside your loop where hydrogen_production is calculated
            fractional_load = (operating_voltage * operating_current) / PEM_capacity

            # Clip fractional load between 0 and 1 to avoid extrapolation
            fractional_load = np.clip(fractional_load, 0, 1)

            # Get the production efficiency from the piecewise curve
            production_efficiency = np.interp(fractional_load, load_points, efficiency_points)

            # Hydrogen production

            if mppt_current>max(pem_current_densities):
              H2_prod= 0
            else:
              H2_prod = (operating_voltage * operating_current * production_efficiency) / self.LHV
            hydro_results.append(H2_prod)
            #print(f"Fractional load: {fractional_load:.2f}, Production efficiency: {production_efficiency:.2f}, Hydrogen production: {H2_prod:.4f} kg")


            # PV area
            pv_area = num_modules_series * num_modules_parallel * self.module_area

            # STH efficiency
            if mppt_current>max(pem_current_densities):
              sth_eff=0
            else:
              sth_eff = (H2_prod * self.LHV) / (self.irradiance * pv_area) if self.irradiance*pv_area>0 else 0
            sth_results.append(sth_eff)

            # Energy loss / economic: total CAPEX
            pv_cost = num_modules_series * num_modules_parallel * self.cost_pv_module
            pem_cost = (num_cell_series-2) * num_cell_parallel * self.cost_pem_cell
            total_cost = pv_cost + pem_cost
            r = 0.07       # discount rate (7%)
            n = 20         # lifetime (years)
            annual_OandM_fixed = 0.02 * total_cost  # example: 2% CAPEX per year
            # Annuity factor
            if r == 0:
                annuity = 1.0 / n
            else:
                annuity = r / (1.0 - (1.0 + r) ** (-n))
            # Annualized CAPEX
            annualized_capex = total_cost * annuity
            annual_H2 = H2_prod * 10*365
            if H2_prod == 0:
                cost_h2 = float('inf')  # or set to np.nan, or any placeholder
            else:
                cost_h2 = (annualized_capex + annual_OandM_fixed) / annual_H2
            cost_results.append(total_cost)

            #print(f"Annualized CAPEX: {annualized_capex:.0f} € / yr")
            #print(f"Annual H2: {annual_H2:.1f} kg/yr")
            #print(f"LCOH: {cost_h2:.3f} €/kg")

            energy_losses = abs(mppt_power - operating_voltage * operating_current) # Use V_op, I_op from operating point
            loss_results.append(energy_losses)



            if hydro_results:
                f1[i] = -np.mean(hydro_results)  # maximize
                f2[i] = -np.mean(sth_results)    # maximize
                f3[i] = np.mean(cost_results)   # minimize cost
                f4[i] = np.mean(loss_results)
            else:
                f1[i] = f2[i] = f3[i] = f4[i] = np.inf

            # Constraint: PV voltage should be >= PEM voltage
            # This constraint needs to be evaluated based on the operating point, not just max currents
            # For simplicity, if an operating point was found, we assume the constraint is met or set it to 0.
            # If no operating point was found (e.g., due to 'continue'), then f1-f4 are inf and g1[i] should be inf as well.
            # If an operating point was found, we set g1[i] = 0 as it's a feasibility constraint.
            if not hydro_results: # If no hydro_results, then it was infeasible
                g1[i] = np.inf
            else:
                g1[i] = 0 # If an operating point was found, the system is considered matched (feasible).

        out["F"] = np.column_stack([f1, f2, f3, f4])
        out["G"] = np.column_stack([g1])

In [30]:
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.termination import get_termination
import pandas as pd
import matplotlib.pyplot as plt


# Read irradiance values from Excel file
df_irradiance = pd.read_excel('Irradiance_values_example.xlsx','Days')

# Initialize an empty list to store all results
all_results_flex = []

# Group the data by day
grouped = df_irradiance.groupby('Day')

for day, group in grouped:

    hourly_averaged_irradiance_values = group['Irradiance_Lund'].tolist()
    hourly_averaged_temperature_values = group['Temperature_Lund'].tolist()

    # Iterate through each hour of the day
    for hour, (irradiance, temperature) in enumerate(zip(hourly_averaged_irradiance_values, hourly_averaged_temperature_values), 1):
        print(f"Processing data for Day {day}, Hour {hour}, Irradiance = {irradiance} W/m², Ambient_T = {temperature}°C")

        if irradiance<80:
            H2 = 0,
            StH = 0,
            Cost=0,
            Loss=0
            continue

        # Define the optimization problem instance for the current hour's conditions
        problem = PV_PEM_Optimization(
            params=params_base, # Changed from params_base + [temperature]
            irradiance=irradiance,
            #T=temperature + 273.15, # Convert to Kelvin
            p_cat=p_cat,
            p_an=p_an,
            δ_mem=δ_mem,
            A_cell_pem=A_cell, # Use A_cell from previous cells
            a_an=a_an,
            a_cat=a_cat,
            i_0_an=i_0_an,
            i_0_cat=i_0_cat,
            T_ref=T_ref,
            i_lim=i_lim,
            LHV=LHV,
            module_area=module_area,
            efficiency=efficiency,
            tilt_angle_deg=tilt_angle_deg,
            sun_zenith_deg=sun_zenith_deg
        )

        # Initialize the NSGA-II algorithm
        algorithm = NSGA2(
            pop_size=12,
            n_offsprings=6,
            sampling=FloatRandomSampling(),
            crossover=SBX(prob=0.9, eta=20),
            mutation=PM(eta=20),
            eliminate_duplicates=True
        )

        # Define termination criteria
        termination = get_termination("n_evals", 20)

        # Run the optimization
        res = minimize(problem,
                       algorithm,
                       termination,
                       #seed=1, # You might want to change the seed or remove it for different runs
                       save_history=False, # No need to save history for each hour
                       verbose=True) # Set to True if you want to see optimization progress for each hour

        # Check if results are available and valid before processing
        if res.F is not None and len(res.F) > 0:
            # Print key results for this hour
            print(f"🔍 Pareto front for Day {day}, Hour {hour}:")
            for i, (f1, f2, f3, f4) in enumerate(res.F):
                H2 = -f1                    # Remember: hydrogen production is minimized as -H2
                StH = -f2
                Cost = f3                  # Same for STH efficiency
                Loss = f4

                config = res.X[i]
                print(f"  Solution {i+1}:")
                print(f"    → Modules (series, parallel): {int(config[0])}, {int(config[1])}")
                print(f"    → Cells (series, parallel):   {int(config[2])}, {int(config[3])}")
                print(f"    → PEM_Temperature:{int(config[4])}°C")
                print(f"    → Hydrogen Production: {H2:.4f} kg")
                print(f"    → STH Efficiency:      {StH:.2%}")
                print(f"    → Economic Cost:       {Cost/1000:.2f} kEuro")
                print(f"    → Energy Losses:       {Loss/1000:.2f} kW")

                num_modules_series = int(config[0])
                num_modules_parallel = int(config[1])
                num_cell_series = int(config[2])
                num_cell_parallel = int(config[3])


                # --- PV IV Curve ---
                effective_irradiance = calculate_effective_irradiance(
                    irradiance, 0.1 * irradiance,
                    tilt_angle_deg,sun_zenith_deg
                )
                voltages_pv, currents_pv, powers_pv, mppt_index = perturb_observe_series(
                    params=params_base + [temperature], # params_base + current T for pv_model
                    effective_irradiance=effective_irradiance,
                    num_modules_series=num_modules_series,
                    num_modules_parallel=num_modules_parallel
                )
                mppt_voltage, mppt_current, mppt_power = find_mppt(voltages_pv, currents_pv)

                # --- Generate PEM IV curve
                pem_current_densities = np.linspace(0.01, I_PEM(i_cell, num_cell_parallel), 100)  # Current density (A/cm²)
                # Use the parameters from the current hour's results (stored in the result dictionary)
                pem_voltages = [V_PEM(temperature + 273.15, p_cat, p_an, δ_mem,
                                      A_cell, a_an, a_cat,
                                      i, i_0_an, i_0_cat,
                                      T_ref, i_lim, num_cell_series)
                                for i in pem_current_densities]



                # --- Find intersection (operating point) ---
                operating_point = find_intersection_point(voltages_pv, currents_pv, pem_voltages, pem_current_densities)
                if not operating_point :
                    H2 = 0,
                    StH = 0,
                    Cost=0,
                    Loss=0


                operating_voltage, operating_current = operating_point
                print(operating_point)

                # --- Plot IV Curves and Annotations ---
                plt.figure(figsize=(8, 6))

                plt.plot(voltages_pv, currents_pv, 'b-', linewidth=2, label='PV IV Curve')
                plt.plot(pem_voltages, pem_current_densities, '--', color='black', linewidth=2, label='PEM IV Curve')

                plt.scatter(mppt_voltage, mppt_current, color='red', s=80, edgecolors='black', zorder=5, label='MPPT Point')
                plt.scatter(operating_voltage, operating_current, color='green', s=80, edgecolors='black', zorder=5, label='Operating Point')

                plt.annotate(f"MPPT\n({mppt_voltage:.1f} V, {mppt_current:.1f} A)",
                            xy=(mppt_voltage, mppt_current),
                            xytext=(mppt_voltage + 20, mppt_current +20),
                            arrowprops=dict(arrowstyle="->", color='black'),
                            fontsize=11, color='red', weight='bold')

                plt.annotate(f"Operating\n({operating_voltage:.1f} V, {operating_current:.1f} A)",
                            xy=(operating_voltage, operating_current),
                            xytext=(operating_voltage - 100, operating_current +40),
                            arrowprops=dict(arrowstyle="->", color='black'),
                            fontsize=11, color='green', weight='bold')

                plt.xlabel('Voltage (V)', fontsize=13, fontweight='bold')
                plt.ylabel('Current (A)', fontsize=13, fontweight='bold')
                plt.title(f"Day {day}, Hour {hour} - Irradiance: {irradiance:.0f} W/m²\n"
                          f"Config: Modules ({num_modules_series}, {num_modules_parallel}), Cells ({num_cell_series}, {num_cell_parallel})",
                          fontsize=13, fontweight='bold')

                #plt.xlim(0, max(max(voltages_pv), max(pem_voltages)) + 10)
                # Set x-axis limit
                x_max = max(max(np.array(voltages_pv)), max(pem_voltages)) + 20
                plt.xlim(0, x_max)

                # Set x-axis ticks every 20 units
                plt.xticks(np.arange(0, x_max + 1, 10))
                plt.ylim(0, max(max(currents_pv), max(pem_current_densities)) + 2)
                plt.grid(True, linestyle='--', alpha=0.7)
                plt.legend(loc='upper left', fontsize=10)
                plt.tight_layout()
                plt.show()


            # Store the results along with the irradiance and temperature values
            all_results_flex.append({
                'day': day,
                'hour': hour,
                'irradiance': irradiance,
                'temperature': temperature,
                'pareto_front': res.F,
                'pareto_solutions': res.X
            })
        else:
            print(f"❌ No feasible solutions found for Day {day}, Hour {hour}.")


print("\nOptimization process completed for all data points.")
# You can now access all_results_flex to analyze the results for each hour
# Example: print the number of results
print(f"Number of optimization results stored: {len(all_results_flex)}")

import json

with open("nsga2_results.json", "w") as f:
    json.dump(all_results_flex, f, indent=2, default=lambda x: x.tolist())

import pandas as pd

summary_data = []
for r in all_results_flex:
    for i, (f1, f2, f3, f4) in enumerate(r['pareto_front']):
        summary_data.append({
            "day": r['day'],
            "hour": r['hour'],
            "irradiance": r['irradiance'],
            "temperature": r['temperature'],
            "hydrogen_kg": -f1,
            "sth_efficiency": -f2,
            "cost": f3,
            "losses": f4,
            "modules_series": int(r['pareto_solutions'][i][0]),
            "modules_parallel": int(r['pareto_solutions'][i][1]),
            "cells_series": int(r['pareto_solutions'][i][2]),
            "cells_parallel": int(r['pareto_solutions'][i][3]),
            "pem_temperature": int(r['pareto_solutions'][i][4])
        })

df_summary = pd.DataFrame(summary_data)
df_summary.to_csv("nsga2_summary_last_dance.csv", index=False)
print("Summary CSV saved to nsga2_summary_last_dance.csv")


from google.colab import files
files.download("nsga2_summary_last_dance.csv")

end_time = time.time()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time/60} mins")




Output hidden; open in https://colab.research.google.com to view.

In [31]:
from google.colab import files
files.download("nsga2_summary_last_dance.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>